In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# -----------------------------
# Configuration and Paths
# -----------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.figsize': (12, 6),
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.format': 'png',
})

BASE_DIR = Path("visualizations")
BASE_DIR.mkdir(exist_ok=True)

DATA_FILE = Path("/content/nepse.json")

# -----------------------------
# Load and Prepare Data
# -----------------------------
with open(DATA_FILE, "r") as f:
    raw = json.load(f)
data = raw["data"] if "data" in raw else raw
df = pd.DataFrame(data)
df["close"] = pd.to_numeric(df["close"], errors="coerce")
df["f_date"] = pd.to_datetime(df["f_date"])
df = df[["f_date", "close"]].sort_values("f_date").dropna().reset_index(drop=True)

df["Log_Return"] = np.log(df["close"] / df["close"].shift(1))
df = df.dropna().reset_index(drop=True)

# Split point (80/20)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# -----------------------------
# Figure 1: NEPSE Price Series
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    df["f_date"],
    df["close"],
    color="black",
    linewidth=1.4,
    label="Closing Price",
)

# Highlight train vs test region (subtle shading)
ax.axvspan(
    train_df["f_date"].iloc[-1],
    df["f_date"].iloc[-1],
    color="tab:green",
    alpha=0.05,
    label="Test Period",
)

ax.set_title("NEPSE Index Closing Price (July 1997 – November 2025)")
ax.set_xlabel("Date")
ax.set_ylabel("Closing Price (NPR)")

format_date_axis(ax)
ax.grid(alpha=0.2)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(BASE_DIR / "nepse_price_series.png")
plt.close(fig)



In [31]:
# -----------------------------
# Figure 2: Log-Returns Distribution
# -----------------------------
plt.figure(figsize=(12, 6))
sns.histplot(df["Log_Return"], kde=True, stat="density", bins=100, color="steelblue", alpha=0.7)
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, df["Log_Return"].mean(), df["Log_Return"].std())
plt.plot(x, p, 'k', linewidth=2, label=f'Normal fit (μ={df["Log_Return"].mean():.5f}, σ={df["Log_Return"].std():.5f})')
plt.title("Distribution of Daily Log-Returns (NEPSE Index)")
plt.xlabel("Log-Return")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "log_returns_distribution.png")
plt.close()

In [32]:


# -----------------------------
# Figure 3: Optimal XGBoost Forecast
# -----------------------------
# Load optimal predictions (expanding_lag20)
pred_file = Path("/content/forecast_expanding_lag20.csv")
if pred_file.exists():
    pred_df = pd.read_csv(pred_file)
    plot_n = 400
    actual = pred_df["Actual_Close"].values[-plot_n:]
    predicted = pred_df["Predicted_Close"].values[-plot_n:]
    dates = pd.to_datetime(pred_df["Date"].values[-plot_n:])

    plt.figure(figsize=(14, 7))
    plt.plot(dates, actual, label="Actual Closing Price", color="black", linewidth=1.8)
    plt.plot(dates, predicted, label="XGBoost Predicted (Expanding, 20 lags)", color="#2E8B57", linewidth=1.8)
    plt.title("Out-of-Sample Forecast: Optimal XGBoost Model (Last 400 Days)")
    plt.xlabel("Date")
    plt.ylabel("Closing Price (NPR)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(BASE_DIR / "forecast_comparison_expanding_lag20.png")
    plt.close()



In [33]:


# -----------------------------
# Figure 5: Feature Importance (Optimal Model)
# -----------------------------
importance_file = Path("/content/importance_expanding_lag20.csv")
if importance_file.exists():
    imp_df = pd.read_csv(importance_file)
    imp_df = imp_df.sort_values("Importance", ascending=True).tail(5)  # Top 5

    plt.figure(figsize=(10, 8))
    sns.barplot(data=imp_df, x="Importance", y="Feature", palette="viridis")
    plt.title("Top 15 Feature Importance by Gain\n(Optimal XGBoost: Expanding Window, 20 Lags)")
    plt.xlabel("Importance (Gain)")
    plt.tight_layout()
    plt.savefig(BASE_DIR / "feature_importance_expanding_lag20.png")
    plt.close()

print("All publication-quality figures generated in:", BASE_DIR)

/tmp/ipython-input-3957341791.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=imp_df, x="Importance", y="Feature", palette="viridis")


All publication-quality figures generated in: visualizations


In [21]:
!rm -rf visualizations/

In [34]:
!zip -r /content/nepse_visualizations.zip /content/visualizations


  adding: content/visualizations/ (stored 0%)
  adding: content/visualizations/nepse_price_series.png (deflated 15%)
  adding: content/visualizations/log_returns_distribution.png (deflated 18%)
  adding: content/visualizations/forecast_comparison_expanding_lag20.png (deflated 11%)
  adding: content/visualizations/feature_importance_expanding_lag20.png (deflated 30%)
